# **Notebook**: Landslide Exposure Database (LED) - Part 1

__Date__: September, 2025  
__Author__: D. Acosta-Reyes  
__Supervisor__: J. Wartman  
__University of Washington__

__Content__: Code utilized to aggregate information at the building-level  
* PART 1: Building Inventory Integration
    * Spatial Join and Nearest-Neighbor Comparison
    * Geometric filters
    * Proximity evaluation
    * Machine Learning Classification
    * Landslide Susceptibility Attribution  

See next notebooks for:
* PART 2: Geographic Classification
    * Physiographic Divisions
    * Census Divisions
    * States
    * Census Tracts
    * Census Blocks
* PART 3: Socioeconomic Attributes
    * Population
    * CRE
    * Poverty and Income
    * Error Metrics


### Initial setup

In [ ]:
'''Load and import necessary libraries'''
# General data libraries
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio

# Utilities
import gc
import warnings
import joblib

# Machine learning
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

import os

# Define data path
data_path = "/Users/danielacosta/Library/CloudStorage/OneDrive-UW/0 - DA General Exam/Paper 1 - Exposure analysis/Data"

In [ ]:
# Subset for analysis:
state = ['CONUS', 'AK', 'HI']

# dictionary of state abreviation for reference -- If a specific state, add next to 'CONUS':
state_dict = {
    'CONUS': 'conus',
    'AK': 'Alaska',
    'HI': 'Hawaii'
}
# 

##### Load datasets: Overture and NSI

In [ ]:
# Load Overture footprints -- CONUS, AK, HI files
conus_file_path = f'overture_buildings/{state[0]}_overture_buildings.gpkg'
ak_file_path = f'overture_buildings/{state[1]}_overture_buildings.gpkg'
hi_file_path = f'overture_buildings/{state[2]}_overture_buildings.gpkg'

conus_overture_fp = gpd.read_file(os.path.join(data_path, conus_file_path))
conus_overture_count = len(conus_overture_fp)

if ak_file_path.exists():
    ak_overture_fp = gpd.read_file(os.path.join(data_path, ak_file_path))
    ak_overture_count = len(ak_overture_fp)
else:
    print(f"Warning: AK Overture file not found at {os.path.join(data_path, ak_file_path)}. Please check the file path and ensure the file is downloaded.")
    ak_overture_fp = gpd.GeoDataFrame()  # Create an empty GeoDataFrame as a placeholder
if hi_file_path.exists():
    hi_overture_fp = gpd.read_file(os.path.join(data_path, hi_file_path))
    hi_overture_count = len(hi_overture_fp)
else:
    print(f"Warning: HI Overture file not found at {os.path.join(data_path, hi_file_path)}. Please check the file path and ensure the file is downloaded.")
    hi_overture_fp = gpd.GeoDataFrame()  # Create an empty GeoDataFrame as a placeholder

# Print confirm data loaded
print(f'CONUS Overture footprints: {len(conus_overture_fp)} records')
print(f'AK Overture footprints: {len(ak_overture_fp)} records')
print(f'HI Overture footprints: {len(hi_overture_fp)} records')
print(conus_overture_fp.columns) # Check and confirm columns in Overture data

In [ ]:
# Load NSI buildings point data -- CONUS, AK, HI files
us_file_path = f'nsi_buildings/US_nsi_2022.gpkg'

us_nsi = gpd.read_file(os.path.join(data_path, us_file_path))
# divide conus, ak, hi data from the full US dataset based on state identifier (STATEFP)
conus_nsi = us_nsi[~us_nsi['STATEFP'].isin(['02', '15'])].copy()  # Exclude AK and HI
ak_nsi_gdf = us_nsi[us_nsi['STATEFP'] == '02'].copy()  # AK
hi_nsi_gdf = us_nsi[us_nsi['STATEFP'] == '15'].copy()  # HI

print(f'CONUS NSI buildings: {len(conus_nsi)} records')
print(f'AK NSI buildings: {len(ak_nsi_gdf)} records')
print(f'HI NSI buildings: {len(hi_nsi_gdf)} records')
print(conus_nsi.columns) # Check and confirm columns in NSI data

#### Define CRS for the project

In [ ]:
# CRS CONUS
conus_crs = 'EPSG:5070' # NAD83 / Conus Albers
ak_crs = 'EPSG:3338' # NAD83 / Alaska Albers
hi_crs = 'EPSG:4135' # NAD83 / Hawaii Albers

## 1) Building Inventory Integration  
_Objective_: Integrate NSI information with Overture Footprints and USGS Landslide Susceptibility. This process includes:
* A) Spatial Join and Nearest-Neighbor Comparison
* B) Geometric filters
* C) Proximity evaluation
* D) Machine Learning Classification
* E) Landslide Susceptibility Attribution


### Pre-Process

In [ ]:
def pre_process(
    conus_nsi,
    conus_overture_fp,
    conus_crs,
    ak_nsi_gdf=None,
    hi_nsi_gdf=None,
    ak_overture_fp=None,
    hi_overture_fp=None,
    ak_crs=None,
    hi_crs=None,
    conus_ov_non_dwell=None,
    ak_ov_non_dwell=None,
    hi_ov_non_dwell=None
):
    """
    Pre-process NSI and Overture data for spatial join.
    Skips AK/HI if not provided, empty, or missing required fields.
    """
    conus_ov_non_dwell = conus_ov_non_dwell or []
    ak_ov_non_dwell = ak_ov_non_dwell or []
    hi_ov_non_dwell = hi_ov_non_dwell or []

    def _is_valid_gdf(gdf):
        return gdf is not None and hasattr(gdf, "empty") and not gdf.empty and "geometry" in gdf.columns

    def _prep_nsi(gdf, target_crs):
        if not _is_valid_gdf(gdf) or "bid" not in gdf.columns:
            return None

        cols = ['bid', 'geometry']
        if 'st_damcat' in gdf.columns:
            cols.insert(1, 'st_damcat')
        out = gdf[cols].copy()

        # Deduplicate NSI by bid; prioritize RES when st_damcat exists
        if 'st_damcat' in out.columns:
            out = (
                out.assign(_is_res=out['st_damcat'].eq('RES'))
                .sort_values(by=['bid', '_is_res'], ascending=[True, False], kind="mergesort")
                .drop_duplicates(subset=['bid'], keep="first")
                .drop(columns=['_is_res'])
            )
        else:
            out = out.drop_duplicates(subset=['bid'], keep="first")

        if target_crs is not None:
            out = out.to_crs(target_crs)
        return out

    def _prep_overture(gdf, target_crs, non_dwell_list):
        if not _is_valid_gdf(gdf) or "class" not in gdf.columns:
            return None, None

        out = gdf[['class', 'STATEFP', 'geometry']].copy()
        out = out.drop_duplicates(subset=['geometry'], keep="first")

        if target_crs is not None:
            out = out.to_crs(target_crs)

        out['geoid'] = out.index.astype(str)

        non_dwell_list = [
        "agricultural", "bridge_structure", "cowshed", "digester", "farm",
        "farm_auxiliary", "garage", "garages", "parking", "silo", "slurry_tank",
        "stable", "storage_tank", "transformer_tower", "wayside_shrine"
        ] 
        # Normalize class values for reliable matching
        class_norm = out['class'].fillna('').astype(str).str.strip().str.lower()
        mask_non_dwell = class_norm.isin(non_dwell_list)

        # Save non-dwell before filtering
        non_dwell = out.loc[mask_non_dwell].copy()

        # Keep dwell-like structures
        out = out.loc[~mask_non_dwell].copy()

        return out, non_dwell

    # CONUS (required)
    conus_nsi_light = _prep_nsi(conus_nsi, conus_crs)
    conus_overture_fp_proc, conus_overture_non_dwell = _prep_overture(
        conus_overture_fp, conus_crs, conus_ov_non_dwell
    )

    # Optional AK
    ak_nsi_light = _prep_nsi(ak_nsi_gdf, ak_crs) if _is_valid_gdf(ak_nsi_gdf) else None
    ak_overture_fp_proc, ak_overture_non_dwell = (
        _prep_overture(ak_overture_fp, ak_crs, ak_ov_non_dwell) if _is_valid_gdf(ak_overture_fp) else (None, None)
    )

    # Optional HI
    hi_nsi_light = _prep_nsi(hi_nsi_gdf, hi_crs) if _is_valid_gdf(hi_nsi_gdf) else None
    hi_overture_fp_proc, hi_overture_non_dwell = (
        _prep_overture(hi_overture_fp, hi_crs, hi_ov_non_dwell) if _is_valid_gdf(hi_overture_fp) else (None, None)
    )

    return {
        "conus_nsi_light": conus_nsi_light,
        "ak_nsi_light": ak_nsi_light,
        "hi_nsi_light": hi_nsi_light,
        "conus_overture_fp": conus_overture_fp_proc,
        "ak_overture_fp": ak_overture_fp_proc,
        "hi_overture_fp": hi_overture_fp_proc,
        "conus_overture_non_dwell": conus_overture_non_dwell,
        "ak_overture_non_dwell": ak_overture_non_dwell,
        "hi_overture_non_dwell": hi_overture_non_dwell,
    }

In [ ]:
'''RUN PRE-PROCESSING'''
# Usage (safe for missing AK/HI variables):
results = pre_process(
    conus_nsi=conus_nsi,
    conus_overture_fp=conus_overture_fp,
    conus_crs=conus_crs,
    ak_nsi_gdf=globals().get("ak_nsi_gdf"),
    hi_nsi_gdf=globals().get("hi_nsi_gdf"),
    ak_overture_fp=globals().get("ak_overture_fp"),
    hi_overture_fp=globals().get("hi_overture_fp"),
    ak_crs=globals().get("ak_crs"),
    hi_crs=globals().get("hi_crs"),
    conus_ov_non_dwell=globals().get("conus_ov_non_dwell", []),
    ak_ov_non_dwell=globals().get("ak_ov_non_dwell", []),
    hi_ov_non_dwell=globals().get("hi_ov_non_dwell", []),
)

conus_nsi_light = results["conus_nsi_light"]
conus_overture_fp = results["conus_overture_fp"]
conus_overture_non_dwell = results["conus_overture_non_dwell"]

ak_nsi_light = results["ak_nsi_light"]
hi_nsi_light = results["hi_nsi_light"]
ak_overture_fp = results["ak_overture_fp"]
hi_overture_fp = results["hi_overture_fp"]
ak_overture_non_dwell = results["ak_overture_non_dwell"]
hi_overture_non_dwell = results["hi_overture_non_dwell"]

# print summary of pre-processing results
print(f"Pre-processing completed:")
print(f"  {state[0]} NSI (light): {len(conus_nsi_light)} records")
print(f"  {state[0]} Overture (dwell-like): {len(conus_overture_fp)} records")
print(f"  {state[0]} Overture (non-dwell): {len(conus_overture_non_dwell)} records")
print(f"  {state[1]} NSI (light): {len(ak_nsi_light) if ak_nsi_light is not None else 'N/A'} records")
print(f"  {state[1]} Overture (dwell-like): {len(ak_overture_fp) if ak_overture_fp is not None else 'N/A'} records")
print(f"  {state[1]} Overture (non-dwell): {len(ak_overture_non_dwell) if ak_overture_non_dwell is not None else 'N/A'} records")
print(f"  {state[2]} NSI (light): {len(hi_nsi_light) if hi_nsi_light is not None else 'N/A'} records")
print(f"  {state[2]} Overture (dwell-like): {len(hi_overture_fp) if hi_overture_fp is not None else 'N/A'} records")
print(f"  {state[2]} Overture (non-dwell): {len(hi_overture_non_dwell) if hi_overture_non_dwell is not None else 'N/A'} records")

### A) Spatial Comparison

In [ ]:
def compare(nsi, overture):
    '''
    Function to process spatial joins
        Inputs: nsi, overture
        Outputs: nsi matched, overture matched, overture near, overture not matched
    '''
    if nsi is None or overture is None:
        print("One or both of the input GeoDataFrames are None. Please check the pre-processing step for issues.")
        return None, None, None, None
    else:
        print(f"Starting spatial join comparison...\n  NSI records: {len(nsi)}\n  Overture records: {len(overture)}")
    # Spatial join NSI building points to Overture building footprints
    if nsi.crs != overture.crs:
        nsi = nsi.to_crs(overture.crs)
    nsi_joined = gpd.sjoin(nsi, overture[['geoid', 'geometry']], how='left', predicate='intersects')
    overture_joined = overture[overture['geoid'].isin(nsi_joined['geoid'])].copy() 
    print(f"Total NSI joined records: {len(nsi_joined[~nsi_joined['geoid'].isna()])}")
    print(f"Total Overture records joined to NSI: {len(overture_joined)}")

    # Get not-joined subset for nsi and overture
    nsi_not_joined = nsi_joined[nsi_joined['geoid'].isna()].copy()
    overture_not_joined = overture[~overture['geoid'].isin(nsi_joined['geoid'])].copy()
    print(f"Total not-joined records:\n  {len(nsi_not_joined)} NSI\n  {len(overture_not_joined)} Overture\nProcess Nearest-Neighbor join for not-joined NSI records...")

    # run nearest neighbor join for not-joined NSI records
    if not nsi_not_joined.empty and not overture.empty:
        nsi_not_joined = nsi_not_joined.to_crs(overture.crs)
        # Use sjoin_nearest with a reasonable max_distance (e.g., 1000 meters) to avoid spurious matches. Drop index_right after join to clean up.
        nsi_not_joined = nsi_not_joined.drop(columns=['index_right','geoid'], errors='ignore')
        nsi_near = gpd.sjoin_nearest(nsi_not_joined, overture[['geoid', 'geometry']], how='left', distance_col='nearest_dist', max_distance=1000)
        print(f"Nearest-Neighbor join completed for not-joined NSI records (max_distance = 1 km)")
        # Get overture not joined and near
        overture_near_joined = overture_not_joined[overture_not_joined['geoid'].isin(nsi_near['geoid'])]
        overture_not_joined = overture_not_joined[~overture_not_joined['geoid'].isin(nsi_near['geoid'])].copy()
        overture_near_nsi = gpd.sjoin_nearest(overture_not_joined, nsi_not_joined[['bid', 'geometry']], how='left', distance_col='nearest_dist', max_distance=1000)
        print(f"Total Overture records near NSI for filtering: {len(overture_near_nsi)}")
        overture_near = pd.concat([overture_near_joined, overture_near_nsi[overture_near_nsi['bid'].notna()]], ignore_index=True)
        print(f"Total Overture records matched to NSI via nearest neighbor join: {len(overture_near)}")
    else:
        print("No records to process for nearest neighbor join (either no not-joined NSI records or Overture data is empty).")
    nsi_matched = pd.concat([nsi_joined[~nsi_joined['geoid'].isna()], nsi_near[~nsi_near['geoid'].isna()]], ignore_index=True)
    print("\n---RESULTS SUMMARY---\n")
    print(f"--> Total NSI records matched to Overture (direct + nearest): {len(nsi_matched)}")
    print(f"--> Total NSI records not matched to Overture: {len(nsi) - len(nsi_matched)}")
    print(f"--> Total Overture records matched to NSI via direct spatial join: {len(overture_joined)}")
    print(f"--> Total Overture records near NSI for filtering: {len(overture_near) if 'overture_near' in locals() else 0}")
    print(f"--> Total Overture records not matched to NSI (Overture Residual): {len(overture) - len(overture_joined) - len(overture_near) if 'overture_near' in locals() else len(overture) - len(overture_joined)}")
    return nsi_matched, overture_joined, overture_near

In [ ]:
'''
Run spatial join comparison
inputs:
    nsi_light, overture_fp
'''
conus_compare = compare(conus_nsi_light, conus_overture_fp)
# Get results from compare function
conus_nsi_matched, conus_overture_joined, conus_overture_near = conus_compare

# Alaska
if 'ak_nsi_light' in globals() and ak_overture_fp is not None and not ak_overture_fp.empty:
    ak_compare = compare(ak_nsi_light, ak_overture_fp)
    ak_nsi_matched, ak_overture_joined, ak_overture_near = ak_compare
else:
    print("AK NSI or Overture data not available for comparison. Skipping AK spatial join.")
    ak_nsi_matched, ak_overture_joined, ak_overture_near = None, None, None

# Hawaii
if 'hi_nsi_light' in globals() and hi_overture_fp is not None and not ak_overture_fp.empty:  
    hi_compare = compare(hi_nsi_light, hi_overture_fp)
    hi_nsi_matched, hi_nsi_not_joined, hi_overture_joined, hi_overture_near = hi_compare
else:
    print("HI NSI or Overture data not available for comparison. Skipping HI spatial join.")
    hi_nsi_matched, hi_nsi_not_joined, hi_overture_joined, hi_overture_near = None, None, None, None

### B) Geometric Filters 

In [ ]:
def geometric_filters(overture_near, overture_non_dwell, bare_min_sqft=400):
    '''
    Unified processing for Overture-near footprints:
      1) Minimum-area filtering (state-based using garage Q25 from non-dwell data)
      2) Shape classification (circular / elongated / complex + hole flag)

    Returns
    -------
    overture_near_filtered : GeoDataFrame
        Kept footprints with added columns:
        ['area', '25th_garage_area_sqm', 'min_area', 'shape_type', 'has_hole']
    overture_residual : GeoDataFrame
        Footprints removed by minimum-area filter
        Footprints removed by circularity and holes
    '''
    if overture_near is None or overture_near.empty:
        print("Overture near data is empty/None. Cannot process.")
        return None, None

    near = overture_near.copy()
    near['area'] = near.geometry.area
    bare_min = bare_min_sqft / 10.764  # sqft -> sqm

    # Build per-state garage 25th percentile area (if available)
    garage_q25_by_state = pd.Series(dtype="float64")
    if (
        overture_non_dwell is not None
        and not overture_non_dwell.empty
        and {'class', 'STATEFP', 'geoid', 'geometry'}.issubset(overture_non_dwell.columns)
    ):
        garage = overture_non_dwell.loc[
            overture_non_dwell['class'].isin(['garage', 'garages']),
            ['class','STATEFP', 'geoid', 'geometry']
        ].copy()

        if not garage.empty:
            garage['area'] = garage.geometry.area
            garage_q25_by_state = garage.groupby('STATEFP')['area'].quantile(0.25)
            print(f"Using state-level garage Q25 thresholds from {len(garage)} garage records.")
        else:
            print("No garage/garages records found in non-dwell data. Using bare minimum only.")
    else:
        print("Non-dwell data is empty/None/missing fields. Using bare minimum only.")

    # Minimum-area threshold per footprint
    near['25th_garage_area_sqm'] = near['STATEFP'].map(garage_q25_by_state)
    near['min_area'] = np.maximum(
        near['25th_garage_area_sqm'].fillna(bare_min).to_numpy(),
        bare_min
    )
    print(f'Minimum area per state is:\n{near[['STATEFP', '25th_garage_area_sqm', 'min_area']].drop_duplicates().to_string(index=False)}')
    # Filter by minimum area
    footprint = near.loc[near['area'] >= near['min_area']].copy()
    # Get count of true garages filtered out by minimum area
    if 'garage' in locals() and not garage.empty:
        garage_min_area = np.maximum(
            garage['STATEFP'].map(garage_q25_by_state).fillna(bare_min).to_numpy(),
            bare_min
        )
        pct_true_garages_filtered = (garage['area'].to_numpy() < garage_min_area).mean() * 100
        print(f"True garages identified by Minimum-Area Filter: {pct_true_garages_filtered:.2f}%")
    else:
        print("True garages identified by Minimum-Area Filter: N/A (no garage records)")
    # get garage records filtered out by minimum area
    if 'garage' in locals() and not garage.empty:
        garage_filtered = garage.loc[garage['area'] < garage_min_area].copy()
        
    # Shape classification helper function
    def _classify_shape(geom):
        if geom is None or geom.is_empty:
            return pd.Series({"shape_type": "complex", "has_hole": False})

        if geom.geom_type == "Polygon":
            has_hole = len(geom.interiors) > 0
        elif geom.geom_type == "MultiPolygon":
            has_hole = any(len(poly.interiors) > 0 for poly in geom.geoms)
        else:
            has_hole = False

        perimeter = geom.length
        if perimeter <= 0:
            shape_type = "complex"
        else:
            circularity = (4.0 * np.pi * geom.area) / (perimeter * perimeter)
            if circularity >= 0.90:
                shape_type = "circular"
            elif circularity <= 0.10:
                shape_type = "elongated"
            else:
                shape_type = "complex"

        return pd.Series({"shape_type": shape_type, "has_hole": has_hole})

    # Apply shape classification to kept footprints
    footprint[['shape_type', 'has_hole']] = footprint.geometry.apply(_classify_shape)

    # Keep only valid shapes: complex and without holes
    has_hole = footprint['has_hole'].fillna(False).astype(bool)
    keep_shape_mask = footprint['shape_type'].eq('complex') & ~has_hole
    true_tanks = overture_non_dwell.loc[overture_non_dwell['class'].isin(['storage_tank', 'slurry_tank', 'silo', 'digester'])].copy()
    true_tanks['shape_type'] = true_tanks.geometry.apply(_classify_shape)['shape_type']
    pct_true_tanks_filtered = (true_tanks['shape_type'].isin(['circular', 'elongated'])).mean() * 100
    print(f"True tanks/silos identified by Shape/Hole Filter: {pct_true_tanks_filtered:.2f}%")
    print(f'Residual after removing hollowed geometries (swimming pools) and circular/elongated structures: {len(footprint.loc[~keep_shape_mask])}')
    # Residuals = removed by minimum-area filter + removed by shape/hole filter
    overture_residual = pd.concat(
        [
            near.loc[near['area'] < near['min_area']],
            footprint.loc[~keep_shape_mask],
        ],
        ignore_index=True,
        copy=False,
    )

    # Final kept footprints (no extra intermediate copy)
    footprint = footprint.loc[keep_shape_mask]
    
    print(f"Total footprints kept: {len(footprint)}")
    print(f"Residual footprints: {len(overture_residual)}")

    return footprint, overture_residual, garage_filtered

In [ ]:
'''
Run geomtric filters on Overture Near data
Main inputs:
    - overture_near
    - overture_non_dwell
Outputs:
    - overture_near_filtered: GeoDataFrame of Overture-near footprints that passed geometric filters, with added columns for area, shape type, and hole flag
    - overture_residual: GeoDataFrame of Overture-near footprints that were filtered out by geometric criteria (either too small, circular/elongated, or have holes)
'''
# Apply geometric filters to Overture-near footprints for each state
conus_overture_near_filtered, conus_residual, garage_filtered = geometric_filters(conus_overture_near, conus_overture_non_dwell)

# if Alaska and Hawaii Overture data were loaded and processed, apply geometric filters; otherwise, set results to None
if 'ak_overture_near' in globals() and ak_overture_near is not None and not ak_overture_near.empty:
    ak_overture_near_filtered, ak_residual, ak_garage_filtered = geometric_filters(ak_overture_near, ak_overture_non_dwell) if ak_overture_near is not None and ak_overture_non_dwell is not None else (None, None, None)
if 'hi_overture_near' in globals() and hi_overture_near is not None and not hi_overture_near.empty:
    hi_overture_near_filtered, hi_residual, hi_garage_filtered = geometric_filters(hi_overture_near, hi_overture_non_dwell) if hi_overture_near is not None and hi_overture_non_dwell is not None else (None, None, None)

### C) Proximity (10 meters)

In [ ]:
'''
Filter overture_near_filtered to keep only those within 10 m of an NSI record.
'''
print(f'mean nearest distance before filtering: {conus_overture_near_filtered["nearest_dist"].mean():.2f} m')
conus_overture_near_final = conus_overture_near_filtered.loc[conus_overture_near_filtered['nearest_dist'] <= 10].copy()
print(f'Footprints within 10 m of NSI records: {len(conus_overture_near_final)}')
# preserve larger than 10 meters for next step
conus_overture_near_far = conus_overture_near_filtered.loc[conus_overture_near_filtered['nearest_dist'] > 10].copy()
print(f'Footprints farther than 10 m from NSI records: {len(conus_overture_near_far)}')

# Alaska and Hawaii -- ignore if data not in system
if 'ak_overture_near_filtered' in globals() and ak_overture_near_filtered is not None and not ak_overture_near_filtered.empty:
    print(f'mean nearest distance before filtering: {ak_overture_near_filtered["nearest_dist"].mean():.2f} m')
    ak_overture_near_final = ak_overture_near_filtered.loc[ak_overture_near_filtered['nearest_dist'] <= 10].copy()
    print(f'AK Footprints within 10 m of NSI records: {len(ak_overture_near_final)}')
    ak_overture_near_far = ak_overture_near_filtered.loc[ak_overture_near_filtered['nearest_dist'] > 10].copy()
    print(f'AK Footprints farther than 10 m from NSI records: {len(ak_overture_near_far)}')
else:
    ak_overture_near_final, ak_overture_near_far = None, None

if 'hi_overture_near_filtered' in globals() and hi_overture_near_filtered is not None and not hi_overture_near_filtered.empty:
    print(f'mean nearest distance before filtering: {hi_overture_near_filtered["nearest_dist"].mean():.2f} m')
    hi_overture_near_final = hi_overture_near_filtered.loc[hi_overture_near_filtered['nearest_dist'] <= 10].copy()
    print(f'HI Footprints within 10 m of NSI records: {len(hi_overture_near_final)}')
    hi_overture_near_far = hi_overture_near_filtered.loc[hi_overture_near_filtered['nearest_dist'] > 10].copy()
    print(f'HI Footprints farther than 10 m from NSI records: {len(hi_overture_near_far)}')
else:
    hi_overture_near_final, hi_overture_near_far = None, None

### D) Machine-Learning Decision Tree Classification

In [ ]:
'''
Machine Learning Classifier: Decision Tree for Garage vs Not-Garage
Training uses:
    - positives: geometrically filtered garage footprints
    - negatives: Overture-near footprints with known non-garage labels
The model predicts unlabeled/non-deterministic records and returns predicted class, confidence, and final label.
'''
def ML_tree_classifier(
    overture_near_far,
    garage_filtered,
    overture_joined,
    feature_cols=['area_ratio_clipped', 'distance_2'],
    max_train_rows=200_000,
    max_ratio=4,
    tree_max_depth=4,
    model_output_path="tree_garage_vs_not_area_ratio_dist.joblib"
):
    """
    Train/apply a binary Decision Tree classifier (garage vs not_garage)
    using area_ratio and distance_2 derived features.
    """
    warnings.filterwarnings("ignore")
    chunk_size = 1_000_000

    # ---- Defensive copies ----
    near_far = overture_near_far.copy()
    garage = garage_filtered.copy()
    joined = overture_joined.copy()

    for df in (near_far, garage, joined):
        if 'index_right' in df.columns:
            df.drop(columns=['index_right'], inplace=True)
    
    # ---- Ensure area columns ----
    if 'area_joined' not in joined.columns:
        joined['area_joined'] = joined.geometry.area
    if 'area' not in garage.columns:
        garage['area'] = garage.geometry.area
    if 'area' not in near_far.columns:
        near_far['area'] = near_far.geometry.area

    # ---- Spatial Joins for distance/area_joined if missing ----
    join_cols = [c for c in ['geoid', 'area_joined', 'geometry'] if c in joined.columns]

    if ('area_joined' not in garage.columns) or ('distance_2' not in garage.columns):
        garage = gpd.sjoin_nearest(garage, joined[join_cols], how='left', distance_col='distance_2', max_distance=100)
        # DROP DUPLICATES FROM TIES
        garage = garage[~garage.index.duplicated(keep='first')]

    if ('area_joined' not in near_far.columns) or ('distance_2' not in near_far.columns):
        near_far = gpd.sjoin_nearest(near_far, joined[join_cols], how='left', distance_col='distance_2', max_distance=1000)
        # DROP DUPLICATES FROM TIES
        near_far = near_far[~near_far.index.duplicated(keep='first')]

    # ---- Feature engineering ----
    for df in (garage, near_far):
        df['area_ratio'] = df['area'] / df['area_joined']
        df['area_ratio'] = df['area_ratio'].replace([np.inf, -np.inf], np.nan)
        df['area_ratio_clipped'] = df['area_ratio'].clip(lower=0, upper=10)

    # ---- Build labels (Original Logic) ----
    # 1. Positives from garage_filtered
    pos = garage[feature_cols].copy()
    pos["label_bin"] = "garage"

    # 2. Normalize near_far labels
    def normalize_label(x):
        if pd.isna(x): return np.nan
        s = str(x).strip().lower()
        if s in {"garage", "garages"}: return "garage"
        if s in ('house','residence','home','dwelling','house_boat'): return 'house'
        if s in ('pool','swimming_pool','water_pool'): return 'pool'
        if s in ('silo','storage_tank','slurry_tank','digester','tank','storage'): return 'tank'
        if s in ('artifact','junk','small_object'): return 'artifact'
        return "other" # explicitly not_garage for binary

    if "class" in near_far.columns:
        near_far["label_norm"] = near_far["class"].apply(normalize_label)
    else:
        near_far["label_norm"] = np.nan

    # Exclude deterministic rules if column exists (matches original logic)
    rule_mask = near_far['rule_label'].isna() if 'rule_label' in near_far.columns else pd.Series(True, index=near_far.index)
    
    # 3. Negatives from near_far (Labeled, but NOT garage, and not excluded by rules)
    known_neg_mask = near_far["label_norm"].notna() & (near_far["label_norm"] != "garage") & rule_mask
    neg = near_far.loc[known_neg_mask, feature_cols].copy()
    neg["label_bin"] = "not_garage"

    # Final training table (garage_filtered doesn't overlap with known_neg_mask)
    train_df_all = pd.concat([pos, neg], ignore_index=True).dropna(subset=feature_cols)

    # ---- Stratified Sampling (Original Logic) ----
    counts = train_df_all['label_bin'].value_counts()
    min_class_count = counts.min()

    parts = []
    for c in counts.index:
        # Cap the majority class at (max_ratio * minority_class_count)
        n_take = min(counts[c], int(max_ratio * min_class_count))
        parts.append(train_df_all[train_df_all['label_bin'] == c].sample(n=n_take, random_state=0))

    # Shuffle the final balanced training set
    train_df = pd.concat(parts).sample(frac=1.0, random_state=0).reset_index(drop=True)

    print("\nTraining distribution after ratio enforcement:")
    print(train_df['label_bin'].value_counts())

    # ---- Train Classifier (Reinstate 'balanced') ----
    train_df = train_df.fillna(0.0)
    X = train_df[feature_cols].values
    y = train_df['label_bin'].values
    clf = DecisionTreeClassifier(max_depth=tree_max_depth, random_state=0)
    clf.fit(X, y)

    # ---- Evaluate on Holdout (Original Logic) ----
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=0)
    clf_eval = DecisionTreeClassifier(max_depth=tree_max_depth, random_state=0)
    clf_eval.fit(X_train, y_train)
    y_pred = clf_eval.predict(X_test)

    print("\nBinary classification report (garage vs not_garage) on holdout:")
    print(classification_report(y_test, y_pred, digits=3))
    
    if 'garage' in clf_eval.classes_:
        try:
            y_prob = clf_eval.predict_proba(X_test)[:, list(clf_eval.classes_).index('garage')]
            roc_auc_holdout = round(roc_auc_score((y_test=='garage').astype(int), y_prob), 3)
            print("ROC AUC (garage vs not_garage) =", roc_auc_holdout)
        except Exception:
            roc_auc_holdout = np.nan

    joblib.dump(clf_eval, model_output_path)

    # Generate a clean, visually appealing confusion matrix plot with descriptive labels and percentages
    # 1. Calculate the raw confusion matrix
    cm = confusion_matrix(y_test, y_pred, labels=['garage', 'not_garage'])

    # 2. Define the descriptive names for each quadrant
    group_names = [
        "True Positives\n(Correct Garages)", 
        "False Negatives\n(Missed Garages)", 
        "False Positives\n(Incorrect Garages)", 
        "True Negatives\n(Correct Non-Garages)"
    ]

    # 3. Get the raw counts
    group_counts = ["{0:0.0f}".format(value) for value in cm.flatten()]

    # 4. Calculate row-wise percentages (How many of the actual class went where?)
    group_percentages = []
    for i in range(2):
        row_sum = cm[i].sum()
        for j in range(2):
            pct = (cm[i, j] / row_sum) * 100 if row_sum > 0 else 0
            group_percentages.append(f"{pct:.1f}%")

    # 5. Combine names, counts, and percentages into the final box labels
    labels = [f"{v1}\n\n{v2}\n({v3})" for v1, v2, v3 in zip(group_names, group_counts, group_percentages)]
    labels = np.asarray(labels).reshape(2, 2)

    # 6. Plot using seaborn
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm, 
        annot=labels, 
        fmt='',             # Tells seaborn to use our custom string labels exactly as written
        cmap='Blues',       # A clean, professional color palette
        cbar=False,         # Hides the color scale bar (unnecessary since we have text)
        xticklabels=['Predicted: Garage', 'Predicted: Not Garage'], 
        yticklabels=['Actual: Garage', 'Actual: Not Garage'],
        annot_kws={"size": 11} # Adjusts font size inside the boxes
    )

    plt.title('Classification Confusion Matrix: Garage vs Not Garage', pad=20, fontsize=14)
    plt.tight_layout()
    plt.show()

    # ---- Apply Classifier (Original Logic) ----
    # Apply to unlabeled AND not-excluded
    apply_mask = near_far['class'].isna() & rule_mask
    apply_idx = near_far.index[apply_mask].to_numpy()
    
    near_far['pred_bin_label'] = None
    near_far['pred_bin_conf'] = np.nan

    if len(apply_idx) > 0:
        for start in range(0, len(apply_idx), chunk_size):
            end = min(start + chunk_size, len(apply_idx))
            idx_chunk = apply_idx[start:end]
            
            X_chunk = near_far.loc[idx_chunk, feature_cols].fillna(0.0).values
            preds_chunk = clf_eval.predict(X_chunk)
            
            if 'garage' in clf_eval.classes_:
                prob_idx = list(clf_eval.classes_).index('garage')
                probs_chunk = clf_eval.predict_proba(X_chunk)[:, prob_idx]
            else:
                probs_chunk = np.zeros(len(preds_chunk))
                
            near_far.loc[idx_chunk, 'pred_bin_label'] = preds_chunk
            near_far.loc[idx_chunk, 'pred_bin_conf'] = probs_chunk

    # Compose final label
    def compose_final(row):
        if 'rule_label' in row and pd.notna(row['rule_label']): return row['rule_label']
        if pd.notna(row.get('pred_bin_label')): return row['pred_bin_label']
        if pd.notna(row.get('label_norm')): return row['label_norm']
        return 'unknown'
        
    near_far['final_label'] = near_far.apply(compose_final, axis=1)

    # Output Metrics Dictionary
    metrics = {
        "n_train": int(len(train_df)),
        "feature_cols_tree": feature_cols,
        "class_counts": train_df["label_bin"].value_counts().to_dict(),
        "model_type": "decision_tree",
        "tree_max_depth": tree_max_depth,
        "roc_auc_holdout_tree": float(roc_auc_holdout) if pd.notna(roc_auc_holdout) else np.nan,
    }

    del X, y, train_df_all, train_df
    gc.collect()

    return clf_eval, metrics, near_far

In [ ]:
'''
Run binary ML classification (garage vs not_garage) on CONUS Overture-near footprints beyond 10 m from NSI.
Training uses:
    - positives: geometrically filtered garage footprints
    - negatives: Overture-near footprints with known non-garage labels
The model predicts unlabeled/non-deterministic records and returns predicted class, confidence, and final label.
'''
ml_results = ML_tree_classifier(
    overture_near_far=conus_overture_near_far,
    garage_filtered=garage_filtered,
    overture_joined=conus_overture_joined,
    feature_cols=['area_ratio_clipped', 'distance_2'],
    max_train_rows=200_000,
    max_ratio=4,
    tree_max_depth=4
)
clf, metrics, overture_classified = ml_results

# Print summary of ML results
print("\n---ML Classification Summary---")
print(f"Total Overture-near records classified: {len(overture_classified)}")
print(f"Predicted as Garage: {(overture_classified['pred_bin_label'] == 'garage').sum()} records")
print(f"Predicted as Not Garage: {(overture_classified['pred_bin_label'] == 'not_garage').sum()} records")

# Get subset of overture_classified predicted as 'not_garage' and with final_label 'other' or 'unknown' for inclusion to final inventory
overture_final_additions = overture_classified.loc[
    (~overture_classified['final_label'].isin(['garage']))
].copy()
print(f"Overture-near records for inclusion in final inventory: {len(overture_final_additions)} records")

### REPORT: Final building classification

In [ ]:
def compile_regional_inventories(
    conus_joined, conus_near_final, conus_far_additions, conus_nsi=None,
    ak_joined=None, ak_near_final=None, ak_far_additions=None, ak_nsi=None,
    hi_joined=None, hi_near_final=None, hi_far_additions=None, hi_nsi=None,
    conus_count=None, ak_count=None, hi_count=None
):
    """
    Compiles and cleans final spatial inventories for CONUS, Alaska, and Hawaii.
    Applies universal column dropping, area calculation, and label finalization.
    """
    
    # Define the universal processing logic as an internal helper function
    def process_region(region_name, joined, near_final, far_additions, nsi=None, count=None):
        if joined is None or joined.empty:
            print(f"{region_name} Overture data not available for final inventory composition.\n")
            return None
        
        # 1. Concatenate the three components
        # before concatenation, add provenance/source labels safely
        joined = joined.copy()
        joined['nsi_source'] = 'nsi_loc' if nsi is not None else pd.NA

        if near_final is None or near_final.empty:
            near_final = joined.iloc[0:0].copy()
        else:
            near_final = near_final.copy()
            near_final['nsi_source'] = 'nsi_near'

        if far_additions is None or far_additions.empty:
            far_additions = joined.iloc[0:0].copy()
        else:
            far_additions = far_additions.copy()
            far_additions['nsi_source'] = 'nsi_avg'
        final_df = pd.concat(
            [joined, near_final, far_additions],
            ignore_index=True,
            copy=False
        )
        
        # 2. Clean up intermediate processing columns
        cols_to_drop = [
            'index_right', 'geoid_left', 'nearest_dist', 'area', '25th_garage_area_sqm', 'min_area',
            'shape_type', 'has_hole', 'geoid_right', 'area_joined', 'distance_2', 
            'area_ratio', 'area_ratio_clipped', 'label_norm', 'pred_bin_label', 'pred_bin_conf'
        ]
        final_df = final_df.drop(columns=cols_to_drop, errors='ignore')
        
        # 3. Calculate final area and ensure no null labels
        final_df['area'] = round(final_df.geometry.area, 4)
        final_df['final_label'] = final_df['final_label'].fillna(final_df['class'].fillna('other'))
        
        # 4. Print standardized summary
        print(f"--- {region_name} Summary ---")
        print(f"  Joined to NSI: {len(joined)}")
        print(f"  Near NSI (filtered): {len(near_final) if near_final is not None else 0}")
        print(f"  Far from NSI (ML-classified): {len(far_additions) if far_additions is not None else 0}")
        print(f"  Total {region_name} final inventory: {len(final_df)} records")
        print(f"  Total residual footprints: {count - len(final_df) if count is not None else 'N/A'} records\n")
        
        return final_df

    # Apply the helper function to all three regions
    conus_final = process_region("CONUS", conus_joined, conus_near_final, conus_far_additions, conus_nsi, conus_count)
    ak_final = process_region("Alaska (AK)", ak_joined, ak_near_final, ak_far_additions, ak_nsi, ak_count)
    hi_final = process_region("Hawaii (HI)", hi_joined, hi_near_final, hi_far_additions, hi_nsi, hi_count)
    
    return conus_final, ak_final, hi_final

In [74]:
'''
Final Inventory Composition
Combine:
    - conus_overture_joined (dwell-like footprints directly joined to NSI)  
    - conus_overture_near_final (dwell-like footprints near NSI that passed geometric filters)
    - overture_final_additions (footprints beyond 10 m from NSI that were not classified as garages by ML)
'''
conus_final, ak_final, hi_final = compile_regional_inventories(
    # CONUS
    conus_joined=conus_overture_joined,
    conus_near_final=conus_overture_near_final,
    conus_far_additions=overture_final_additions,
    conus_nsi=conus_nsi_matched,
    conus_count=conus_overture_count,
    
    # Alaska (Pass your pre-filtered ML data here)
    ak_joined=ak_overture_joined if 'ak_overture_joined' in locals() else None,
    ak_near_final=ak_overture_near_final if 'ak_overture_near_final' in locals() else None,
    ak_far_additions=ak_overture_near_far if 'ak_overture_near_far' in locals() else None,
    ak_nsi=None, # Update this if you have an ak_nsi_matched dataframe
    ak_count=ak_overture_count if 'ak_overture_count' in locals() else None,
    
    # Hawaii (Pass your pre-filtered ML data here)
    hi_joined=hi_overture_joined if 'hi_overture_joined' in locals() else None,
    hi_near_final=hi_overture_near_final if 'hi_overture_near_final' in locals() else None,
    hi_far_additions=hi_overture_near_far if 'hi_overture_near_far' in locals() else None,
    hi_nsi=None,  # Update this if you have an hi_nsi_matched dataframe
    hi_count=hi_overture_count if 'hi_overture_count' in locals() else None
)

--- CONUS Summary ---
  Joined to NSI: 2526701
  Near NSI (filtered): 10068
  Far from NSI (ML-classified): 598921
  Total CONUS final inventory: 3135690 records
  Total residual footprints: 685783 records

Alaska (AK) Overture data not available for final inventory composition.

Hawaii (HI) Overture data not available for final inventory composition.



### E) Landslide Susceptibility Attribution

In [ ]:
# Load raster libraries for zonal stats
import rasterio
from rasterio import features
from rasterio.mask import mask

In [ ]:
# load usgs_n10 raster for zonal stats with rasterio

raster_path = os.path.join(data_path, "usgs_n10_susc/n10_susc")

# conus raster
try:
    n10_conus = rasterio.open(raster_path + "/n10_conus.tif")
    print("CONUS N10 raster loaded successfully.")
except Exception as e:
    print(f"Error loading CONUS N10 raster: {e}")
    n10_conus = None

if 'ak_final' in locals() and ak_final is not None:
    # alaska raster
    try:
        n10_ak = rasterio.open(raster_path + "/n10_ak.tif")
        print("Alaska N10 raster loaded successfully.")
    except Exception as e:
        print(f"Error loading Alaska N10 raster: {e}")
        n10_ak = None

if 'hi_final' in locals() and hi_final is not None:
    # hawaii raster
    try:
        n10_hi = rasterio.open(raster_path + "/n10_hi.tif")
        print("Hawaii N10 raster loaded successfully.")
    except Exception as e:
        print(f"Error loading Hawaii N10 raster: {e}")
        n10_hi = None

In [ ]:
# get crs from rasters for later reprojection
raster_crs = None
if n10_conus is not None:
    raster_crs = n10_conus.crs
elif n10_ak is not None:
    raster_crs = n10_ak.crs
elif n10_hi is not None:
    raster_crs = n10_hi.crs
print(f"Raster CRS for zonal stats: {raster_crs}")

# Define a function to perform zonal statistics for a given GeoDataFrame and raster
def perform_zonal_stats(gdf, raster, region_name):
    if gdf is None or gdf.empty:
        print(f"{region_name} GeoDataFrame is empty. Skipping zonal stats.")
        return gdf

    if raster is None:
        print(f"{region_name} raster is not available. Skipping zonal stats.")
        return gdf

    # Reproject GeoDataFrame to match raster CRS if needed
    if gdf.crs != raster.crs:
        print(f"Reprojecting {region_name} GeoDataFrame to match raster CRS for zonal stats.")
        gdf = gdf.to_crs(raster.crs)

    # Prepare a list to hold zonal stats results
    zonal_stats_results = []

    # Loop through each geometry in the GeoDataFrame
    for idx, row in gdf.iterrows():
        geom = row['geometry']
        try:
            # Mask the raster with the geometry
            out_image, out_transform = mask(raster, [geom], crop=True)
            out_image = out_image[0]  # Assuming single band raster

            # Calculate zonal statistics: maximum value
            valid_pixels = out_image[out_image != raster.nodata]
            max_val = valid_pixels.max() if len(valid_pixels) > 0 else np.nan
            zonal_stats_results.append({
                'usgsN10_max': max_val
            })
        except Exception as e:
            print(f"Error processing geometry at index {idx} in {region_name}: {e}")
            zonal_stats_results.append({
                'usgsN10_max': np.nan
            })

    # Convert results to DataFrame and merge back to original GeoDataFrame
    stats_df = pd.DataFrame(zonal_stats_results).set_index('index')
    gdf_with_stats = gdf.join(stats_df)

    return gdf_with_stats
# Perform zonal statistics for each region
conus_final_stats = perform_zonal_stats(conus_final, n10_conus, "CONUS")
if 'ak_final' in locals() and ak_final is not None:
    ak_final_stats = perform_zonal_stats(ak_final, n10_ak, "Alaska")
if 'hi_final' in locals() and hi_final is not None:
    hi_final_stats = perform_zonal_stats(hi_final, n10_hi, "Hawaii")

#### Save data for next steps

In [ ]:
# merge all active inventories into one for export
out_dir = os.path.join(data_path, "process_building_inventory")
os.makedirs(out_dir, exist_ok=True)

# unify crs before merging if needed
crs = 'EPSG:4269'  # NAD83
if conus_final_stats is not None and conus_final_stats.crs != crs:
    conus_final_stats = conus_final_stats.to_crs(crs)
if 'ak_final_stats' in locals() and ak_final_stats is not None and ak_final_stats.crs != crs:
    ak_final_stats = ak_final_stats.to_crs(crs)
if 'hi_final_stats' in locals() and hi_final_stats is not None and hi_final_stats.crs != crs:
    hi_final_stats = hi_final_stats.to_crs(crs)

# merge inventories, ignoring None values for regions that were not processed
final_inventory = pd.concat(
    [conus_final_stats, 
     ak_final_stats if 'ak_final_stats' in locals() else None, 
     hi_final_stats if 'hi_final_stats' in locals() else None],
    ignore_index=True,
    copy=False
)

final_inventory.to_file(os.path.join(output_path, "final_overture_inventory.gpkg"), driver="GPKG")

# convert final inventory to points and reproject to NAD83 for light export
final_inventory_points = final_inventory.copy()
final_inventory_points['geometry'] = final_inventory_points.geometry.centroid
final_inventory_points.to_file(os.path.join(output_path, "final_overture_inventory_points.gpkg"), driver="GPKG")

# save nsi_matched points for reference
# merge all active nsi_matched points into one for export
nsi_matched_points = pd.concat(
    [conus_nsi_matched if 'conus_nsi_matched' in locals() and conus_nsi_matched is not None else None,
     ak_nsi_matched if 'ak_nsi_matched' in locals() and ak_nsi_matched is not None else None,
     hi_nsi_matched if 'hi_nsi_matched' in locals() and hi_nsi_matched is not None else None],
    ignore_index=True,
    copy=False
)

# update nsi_matched_points to include only those with match geoid from final_inventory.
if nsi_matched_points is not None and not nsi_matched_points.empty:
    matched_geoids = set(final_inventory['geoid'].dropna().unique())
    nsi_matched_points = nsi_matched_points[nsi_matched_points['geoid'].isin(matched_geoids)].copy()
    print(f"NSI matched points with geoids in final inventory: {len(nsi_matched_points)} records")

# Reproject nsi_matched_points to NAD83 if needed
if nsi_matched_points is not None and not nsi_matched_points.empty:
    if nsi_matched_points.crs != crs:
        nsi_matched_points = nsi_matched_points.to_crs(crs)
        print(f"Reprojected NSI matched points to {crs} for export.")

# save nsi_matched points to file
if nsi_matched_points is not None and not nsi_matched_points.empty:
    nsi_matched_points.to_file(os.path.join(output_path, "nsi_matched_points.gpkg"), driver="GPKG")

### End of Script